In [1]:
!pip install -q flask pyngrok
!pip install --upgrade transformers accelerate peft bitsandbytes
!pip install git+https://github.com/huggingface/transformers.git -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 80.1 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 87.1 MB/s eta 0:00:00:00:01
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.12.0
    Uninstalling accelerate-1.12.0:
      Successfully uninstalled acc

In [ ]:
import os
import re
import json
import hashlib
import time
import gc
import torch
from flask import Flask, request, jsonify
from pyngrok import ngrok
import threading
from pathlib import Path
from typing import Optional
from dataclasses import dataclass, field, asdict
from enum import Enum
from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig, AutoConfig
import uuid
import traceback
from concurrent.futures import ThreadPoolExecutor

FAST_MODE       = True
OFFLOAD_LM_HEAD = True


# ═════════════════════════════════════════════
# PART A — CACHE LAYER
# ═════════════════════════════════════════════

@dataclass
class CachedUI:
    cache_id:      str
    prompt:        str
    keywords:      list
    ui_target:     str
    concept:       dict
    html_fragment: str
    css_tokens:    str
    js_modules:    str
    score:         int
    created_at:    float = field(default_factory=time.time)
    hit_count:     int   = 0

    def age_hours(self) -> float:
        return (time.time() - self.created_at) / 3600


@dataclass
class CacheMatch:
    found:      bool
    entry:      Optional[CachedUI] = None
    similarity: float              = 0.0
    guide_type: str                = "none"   # "full" | "concept" | "component"


class UICache:
    FULL_THRESHOLD      = 0.85
    CONCEPT_THRESHOLD   = 0.55
    COMPONENT_THRESHOLD = 0.30
    MAX_ENTRIES         = 200
    CACHE_VERSION       = "1.0"

    def __init__(self, cache_dir: str = "./cache"):
        self.cache_dir  = Path(cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.index_path = self.cache_dir / "index.json"
        self._entries: dict[str, CachedUI] = {}
        self._load()

    def _load(self):
        if self.index_path.exists():
            try:
                raw = json.loads(self.index_path.read_text(encoding="utf-8"))
                for cid, data in raw.get("entries", {}).items():
                    self._entries[cid] = CachedUI(**data)
                print(f"[Cache] Loaded {len(self._entries)} entries from {self.index_path}")
            except Exception as e:
                print(f"[Cache] Warning: could not load cache — {e}")

    def _save(self):
        payload = {
            "version":  self.CACHE_VERSION,
            "saved_at": time.time(),
            "entries":  {cid: asdict(e) for cid, e in self._entries.items()},
        }
        self.index_path.write_text(
            json.dumps(payload, indent=2, ensure_ascii=False),
            encoding="utf-8",
        )

    _STOP = {
        "a", "an", "the", "and", "or", "but", "for", "with", "that",
        "this", "it", "is", "are", "be", "to", "of", "in", "on", "at",
        "i", "want", "need", "build", "create", "make", "generate",
        "design", "show", "like", "use", "using", "can", "should",
        "please", "app", "ui", "ux",
    }

    @staticmethod
    def _normalise(prompt: str) -> str:
        return re.sub(r"\s+", " ", prompt.lower().strip())

    def _keywords(self, prompt: str) -> list:
        tokens = re.findall(r"[a-z][a-z0-9\-]*", self._normalise(prompt))
        return sorted({t for t in tokens if t not in self._STOP and len(t) > 2})

    @staticmethod
    def _sha(text: str) -> str:
        return hashlib.sha256(text.encode()).hexdigest()[:16]

    def _cache_id(self, prompt: str) -> str:
        return self._sha(self._normalise(prompt))

    def _jaccard(self, a: list, b: list) -> float:
        sa, sb = set(a), set(b)
        if not sa and not sb:
            return 1.0
        intersection = len(sa & sb)
        union        = len(sa | sb)
        return intersection / union if union else 0.0

    def _best_match(self, keywords: list, ui_target: str) -> CacheMatch:
        best_sim   = 0.0
        best_entry = None
        for entry in self._entries.values():
            sim = self._jaccard(keywords, entry.keywords)
            if entry.ui_target != ui_target:
                sim *= 0.80
            if sim > best_sim:
                best_sim   = sim
                best_entry = entry
        if best_entry is None or best_sim < self.COMPONENT_THRESHOLD:
            return CacheMatch(found=False, similarity=best_sim)
        if   best_sim >= self.FULL_THRESHOLD:    guide = "full"
        elif best_sim >= self.CONCEPT_THRESHOLD: guide = "concept"
        else:                                     guide = "component"
        return CacheMatch(found=True, entry=best_entry, similarity=best_sim, guide_type=guide)

    def lookup(self, prompt: str, ui_target: str) -> CacheMatch:
        cid = self._cache_id(prompt)
        if cid in self._entries:
            entry = self._entries[cid]
            if entry.ui_target == ui_target:
                entry.hit_count += 1
                self._save()
                print(f"[Cache] ✓ Exact hit  (id={cid}, score={entry.score}, hits={entry.hit_count})")
                return CacheMatch(found=True, entry=entry, similarity=1.0, guide_type="full")
        keywords = self._keywords(prompt)
        match    = self._best_match(keywords, ui_target)
        if match.found:
            match.entry.hit_count += 1
            self._save()
            print(f"[Cache] ✓ Fuzzy hit  (id={match.entry.cache_id}, sim={match.similarity:.2f}, guide={match.guide_type})")
        else:
            print(f"[Cache] ✗ Miss  (best_sim={match.similarity:.2f})")
        return match

    def store(self, prompt: str, ui_target: str, concept: dict, final_html: str, audit: dict):
        entry = CachedUI(
            cache_id      = self._cache_id(prompt),
            prompt        = prompt,
            keywords      = self._keywords(prompt),
            ui_target     = ui_target,
            concept       = concept,
            html_fragment = self._extract_html_fragment(final_html),
            css_tokens    = self._extract_css_tokens(final_html),
            js_modules    = self._extract_js_modules(final_html),
            score         = audit.get("score", 0),
        )
        if len(self._entries) >= self.MAX_ENTRIES:
            victim = min(self._entries.values(), key=lambda e: (e.hit_count, e.score))
            del self._entries[victim.cache_id]
            print(f"[Cache] Evicted entry {victim.cache_id} (score={victim.score})")
        self._entries[entry.cache_id] = entry
        self._save()
        print(f"[Cache] ✓ Stored  (id={entry.cache_id}, score={entry.score})")

    def stats(self) -> dict:
        entries = list(self._entries.values())
        return {
            "total_entries":  len(entries),
            "avg_score":      round(sum(e.score for e in entries) / len(entries), 1) if entries else 0,
            "total_hits":     sum(e.hit_count for e in entries),
            "web_entries":    sum(1 for e in entries if e.ui_target == "web"),
            "mobile_entries": sum(1 for e in entries if e.ui_target == "mobile"),
            "oldest_hours":   round(max((e.age_hours() for e in entries), default=0), 1),
        }

    def clear(self):
        self._entries.clear()
        self._save()
        print("[Cache] Cleared.")

    @staticmethod
    def _extract_html_fragment(html: str) -> str:
        m = re.search(r"<body[^>]*>(.*?)</body>", html, re.DOTALL | re.IGNORECASE)
        if m:
            body = m.group(1)
            body = re.sub(r"<style[^>]*>.*?</style>",  "", body, flags=re.DOTALL | re.IGNORECASE)
            body = re.sub(r"<script[^>]*>.*?</script>", "", body, flags=re.DOTALL | re.IGNORECASE)
            body = re.sub(r"\n\s*\n", "\n", body).strip()
            return body[:6000]
        return ""

    @staticmethod
    def _extract_css_tokens(html: str) -> str:
        m = re.search(r":root\s*\{[^}]*\}", html, re.DOTALL)
        return m.group(0).strip() if m else ""

    @staticmethod
    def _extract_js_modules(html: str) -> str:
        m = re.search(r"<script[^>]*>(.*?)</script>", html, re.DOTALL | re.IGNORECASE)
        if not m:
            return ""
        js   = m.group(1)
        sigs = [
            line.strip()
            for line in js.splitlines()
            if re.match(r"(const|function|let)\s+\w+\s*(=\s*\(|\()", line.strip())
        ]
        return "\n".join(sigs[:30])


# ─────────────────────────────────────────────
# Guide Injection Helpers
# ─────────────────────────────────────────────

def build_concept_guide(match: CacheMatch) -> str:
    if not match.found or match.guide_type == "none":
        return ""
    entry = match.entry
    return f"""
── CACHE GUIDE (similarity={match.similarity:.0%}) ──────────────────────────
A previous generation with a similar prompt achieved a score of {entry.score}/100.
Use the following app concept as a STARTING POINT and adapt it to the new prompt.
Do NOT copy it verbatim — keep the spirit, adjust for the new request.

Previous concept:
{json.dumps(entry.concept, indent=2)}
──────────────────────────────────────────────────────────────────────────────
"""


def build_html_guide(match: CacheMatch) -> str:
    if not match.found or not match.entry.html_fragment:
        return ""
    entry = match.entry
    return f"""
── CACHE GUIDE — HTML Structure (sim={match.similarity:.0%}) ─────────────────
The layout below was generated previously for a similar prompt (score={entry.score}/100).
Use it as a STRUCTURAL TEMPLATE — reuse the element hierarchy, class names, and
ARIA roles, but adapt content, copy, and features to the new app concept.

{entry.html_fragment}
──────────────────────────────────────────────────────────────────────────────
"""


def build_css_guide(match: CacheMatch) -> str:
    if not match.found or not match.entry.css_tokens:
        return ""
    entry = match.entry
    return f"""
── CACHE GUIDE — CSS Design Tokens (sim={match.similarity:.0%}) ──────────────
Override only the colour values below to match the new app's palette.
Keep spacing, typography, and animation tokens unless the new concept demands change.

{entry.css_tokens}
──────────────────────────────────────────────────────────────────────────────
"""


def build_js_guide(match: CacheMatch) -> str:
    if not match.found or not match.entry.js_modules:
        return ""
    entry = match.entry
    return f"""
── CACHE GUIDE — JS Module Signatures (sim={match.similarity:.0%}) ───────────
Implement the same module structure as the previous generation.
These are SIGNATURES ONLY — write fresh implementations.

{entry.js_modules}
──────────────────────────────────────────────────────────────────────────────
"""


# ═════════════════════════════════════════════
# PART B — MODEL + PIPELINE
# ═════════════════════════════════════════════

class UITarget(Enum):
    WEB    = "web"
    MOBILE = "mobile"


os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

MODEL_ID = "cihangir10/page.ui"

print("Loading processor…")
processor = AutoProcessor.from_pretrained(MODEL_ID)

print("Loading model…")
model_config = AutoConfig.from_pretrained(MODEL_ID)
print("Model's built-in quant config:", getattr(model_config, "quantization_config", "none"))

if hasattr(model_config, "quantization_config"):
    model_config.quantization_config["llm_int8_enable_fp32_cpu_offload"] = True

device_map = {
    "model":   "cuda:0",
    "lm_head": "cpu",
}

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    config=model_config,
    device_map=device_map,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)
model.config.use_cache = True
print("Model ready ✓")


# ─────────────────────────────────────────────
# Inference Helpers
# ─────────────────────────────────────────────

def call_model(
    system_prompt:  str,
    user_prompt:    str,
    max_new_tokens: int   = 7000,
    temperature:    float = 0.65,
    top_p:          float = 0.92,
) -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ]
    text      = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs    = processor(text=text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=1.1,
            pad_token_id=processor.tokenizer.eos_token_id,
        )
    return processor.decode(outputs[0][input_len:], skip_special_tokens=True).strip()


def call_title_model(
    system_prompt:  str,
    user_prompt:    str,
    max_new_tokens: int   = 20,
    temperature:    float = 0.1,
    top_p:          float = 0.9,
) -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt",
    ).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=1.05,
            do_sample=True,
        )
    return processor.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True).strip()


def extract_section(raw: str, tag: str) -> str:
    pattern = rf"```{tag}\s*(.*?)```"
    match   = re.search(pattern, raw, re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(1).strip()
    pattern2 = rf"###\s*{tag}\s*\n(.*?)(?=###|\Z)"
    match2   = re.search(pattern2, raw, re.DOTALL | re.IGNORECASE)
    if match2:
        return match2.group(1).strip()
    return raw.strip()


def extract_json(raw: str) -> dict:
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            pass
    return {"raw": raw}


# ─────────────────────────────────────────────
# ★ RICH SYSTEM PROMPTS (restored from v1)
# ─────────────────────────────────────────────

TITLE_SYSTEM = """
Generate a short, specific title that captures the key intent of the user's prompt.

STRICT RULES:
- Return ONLY the title — no quotes, no punctuation, no explanation
- Must be between 2 and 4 words (never fewer, never more)
- Be descriptive and specific to the prompt's subject
- Avoid filler words (e.g. "a", "the", "and") unless grammatically essential
- Avoid generic AI clichés like "Exclusive, Trustworthy, Innovative"
- Focus on the core subject and purpose

EXAMPLES:
  Prompt: "Build a luxury watch e-commerce site"       → Luxury Watch Store
  Prompt: "CLI-themed UI generator for developers"     → Terminal UI Generator
  Prompt: "Mobile fitness tracker with AI coaching"    → AI Fitness Tracker
  Prompt: "Dashboard for SaaS analytics platform"      → SaaS Analytics Dashboard
"""

IDEATION_SYSTEM = """
You are a Creative Product Strategist and App Concept Designer (2026).
Your job is to generate a unique, innovative, and market-ready app concept.

Return ONLY valid JSON with this exact structure:
{
  "app_name": "string",
  "tagline": "string (one sentence)",
  "target_audience": "string",
  "problem": "string",
  "features": ["feature1", "feature2", "feature3", "feature4", "feature5"],
  "monetization": "string",
  "user_flow": ["step1", "step2", "step3", "step4", "step5"],
  "ui_style": "string (e.g. dark glassmorphism with neon accents)",
  "color_palette": {
    "primary": "#hex",
    "secondary": "#hex",
    "background": "#hex",
    "surface": "#hex",
    "text": "#hex",
    "accent": "#hex"
  },
  "tech_stack": {
    "frontend": "string",
    "backend": "string",
    "database": "string",
    "ai_features": "string"
  },
  "differentiator": "string"
}

The app must NOT be a clone of Instagram, Uber, TikTok, or WhatsApp.
Be creative, specific, and market-aware.
Return ONLY the JSON object — no preamble, no explanation.
"""

HTML_SYSTEM_WEB = """
You are a Senior Frontend Architect and HTML5 Specialist (2026).

Generate clean, semantic, production-ready HTML5 markup for a FULL-BROWSER WEB UI.

STRICT RULES:
- Use semantic HTML5 elements (header, main, section, nav, footer, article, aside)
- Never use inline styles or inline JavaScript
- Never use deprecated patterns
- Use proper ARIA roles (role, aria-label, aria-expanded, aria-controls)
- Use <button> for ALL interactive actions — never clickable <div>
- Heading hierarchy must be logical (h1 → h2 → h3)
- All images must use real placeholder URLs:
    • https://picsum.photos/{w}/{h}?random={id}
    • https://i.pravatar.cc/{size}?img={id}
    • https://loremflickr.com/{w}/{h}/{keyword}
- Layout structure for a web app:
  <div class="app-shell">
    <header class="topbar" role="banner"> … </header>
    <div class="workspace">
      <aside class="sidebar" role="complementary"> … </aside>
      <main class="main-panel" role="main"> … </main>
    </div>
    <footer class="statusbar" role="contentinfo"> … </footer>
  </div>
- Include Tailwind CDN in <head>:
  <script src="https://cdn.tailwindcss.com"></script>
- Include Google Fonts link relevant to the app's style
- Reference <link rel="stylesheet" href="styles.css">
- Reference <script src="app.js" defer></script>
- SVG icons only — no emoji icons in production UI
- Use data-* attributes for JS hooks, not classes or IDs
- No iPhone frame — this is a desktop/browser layout

OUTPUT: Return ONLY the ### HTML section wrapped in ```html fences.
"""

HTML_SYSTEM_MOBILE = """
You are a Senior Frontend Architect and HTML5 Specialist (2026).

Generate clean, semantic, production-ready HTML5 markup for a mobile app UI.

STRICT RULES:
- Use semantic HTML5 elements (header, main, section, nav, footer, article, aside)
- Never use inline styles or inline JavaScript
- Never use deprecated patterns
- Use proper ARIA roles (role, aria-label, aria-expanded, aria-controls)
- Use <button> for ALL interactive actions — never clickable <div>
- Heading hierarchy must be logical (h1 → h2 → h3)
- All images must use real placeholder URLs:
    • https://picsum.photos/{w}/{h}?random={id}
    • https://i.pravatar.cc/{size}?img={id}
    • https://loremflickr.com/{w}/{h}/{keyword}
- Use the iPhone frame structure exactly:
  <div class="iphone">
    <div class="screen">
      <div class="status-bar">
        <span id="currentTime">9:41</span>
        <div class="status-icons" aria-hidden="true">
          <svg …/>  <!-- signal -->
          <svg …/>  <!-- wifi -->
          <svg …/>  <!-- battery -->
        </div>
      </div>
      <div class="content" role="main">
        <!-- screen content here -->
      </div>
    </div>
  </div>
- Include Tailwind CDN in <head>
- Reference <link rel="stylesheet" href="styles.css">
- Reference <script src="app.js" defer></script>
- SVG icons only — no emoji icons in production UI
- Use data-* attributes for JS hooks

OUTPUT: Return ONLY the ### HTML section wrapped in ```html fences.
"""

CSS_SYSTEM_WEB = """
You are a Senior UI/UX Engineer and Design Systems Specialist (2026).

Generate modern, production-ready CSS for a FULL-BROWSER WEB APP UI.

DESIGN TOKEN RULES (always include full :root block):
:root {
  --color-primary: #7c3aed; --color-secondary: #10b981;
  --color-surface: #1e1e2e; --color-background: #0f0f1a;
  --color-text: #e2e8f0; --color-muted: #64748b; --color-accent: #f59e0b;
  --space-xs:4px; --space-sm:8px; --space-md:16px; --space-lg:24px;
  --space-xl:40px; --space-2xl:64px;
  --radius-sm:8px; --radius-md:16px; --radius-lg:24px; --radius-full:9999px;
  --shadow-soft:0 10px 30px rgba(0,0,0,0.25);
  --shadow-elevated:0 25px 60px rgba(0,0,0,0.40);
  --shadow-glow:0 0 20px var(--color-primary);
  --font-fluid-sm:clamp(0.75rem,1vw,0.9rem);
  --font-fluid-md:clamp(0.95rem,1.4vw,1.1rem);
  --font-fluid-lg:clamp(1.2rem,2vw,1.6rem);
  --font-fluid-xl:clamp(1.8rem,3vw,2.4rem);
  --anim-fast:150ms cubic-bezier(0.4,0,0.2,1);
  --anim-smooth:300ms cubic-bezier(0.4,0,0.2,1);
  --anim-slow:500ms cubic-bezier(0.4,0,0.2,1);
  --glass-blur:20px; --glass-bg:rgba(255,255,255,0.08);
  --glass-border:rgba(255,255,255,0.12);
}

WEB APP SHELL (full viewport):
*, *::before, *::after { box-sizing:border-box; margin:0; padding:0; }
html, body { height:100%; }
body { background:var(--color-background); color:var(--color-text);
       font-family:var(--font-body,sans-serif); overflow:hidden; }
.app-shell { display:flex; flex-direction:column; height:100vh; width:100vw; }
.topbar { height:56px; display:flex; align-items:center; padding:0 var(--space-lg);
          background:var(--color-surface); border-bottom:1px solid var(--glass-border);
          flex-shrink:0; }
.workspace { display:flex; flex:1; overflow:hidden; }
.sidebar { width:240px; background:var(--color-surface);
           border-right:1px solid var(--glass-border);
           overflow-y:auto; flex-shrink:0; padding:var(--space-md); }
.main-panel { flex:1; overflow-y:auto; padding:var(--space-lg); }
.statusbar { height:32px; display:flex; align-items:center; padding:0 var(--space-lg);
             background:var(--color-surface); border-top:1px solid var(--glass-border);
             font-size:var(--font-fluid-sm); color:var(--color-muted); flex-shrink:0; }

GLASS CARD + BUTTON (always include):
.glass-card { background:var(--glass-bg); backdrop-filter:blur(var(--glass-blur));
              border:1px solid var(--glass-border); border-radius:var(--radius-md);
              padding:var(--space-lg); transition:transform var(--anim-smooth),
              box-shadow var(--anim-smooth); }
.glass-card:hover { transform:translateY(-2px); box-shadow:var(--shadow-elevated); }
.btn-primary { background:var(--color-primary); color:#fff; border:none;
               border-radius:var(--radius-sm); padding:var(--space-sm) var(--space-md);
               font-size:var(--font-fluid-sm); cursor:pointer;
               transition:opacity var(--anim-fast), transform var(--anim-fast); }
.btn-primary:hover  { opacity:0.9; transform:translateY(-1px); }
.btn-primary:active { opacity:1;   transform:translateY(0); }

OUTPUT: Return ONLY the ### CSS section wrapped in ```css fences.
"""

CSS_SYSTEM_MOBILE = """
You are a Senior UI/UX Engineer and Design Systems Specialist (2026).

Generate modern, production-ready CSS for a mobile app UI inside an iPhone frame.

DESIGN TOKEN RULES (always include full :root block):
:root {
  --color-primary: #7c3aed; --color-secondary: #10b981;
  --color-surface: #1e1e2e; --color-background: #0f0f1a;
  --color-text: #e2e8f0; --color-muted: #64748b; --color-accent: #f59e0b;
  --space-xs:4px; --space-sm:8px; --space-md:16px; --space-lg:24px;
  --space-xl:40px; --space-2xl:64px;
  --radius-sm:8px; --radius-md:16px; --radius-lg:24px; --radius-full:9999px;
  --shadow-soft:0 10px 30px rgba(0,0,0,0.25);
  --shadow-elevated:0 25px 60px rgba(0,0,0,0.40);
  --shadow-glow:0 0 20px var(--color-primary);
  --font-fluid-sm:clamp(0.75rem,1vw,0.9rem);
  --font-fluid-md:clamp(0.95rem,1.4vw,1.1rem);
  --font-fluid-lg:clamp(1.2rem,2vw,1.6rem);
  --font-fluid-xl:clamp(1.8rem,3vw,2.4rem);
  --anim-fast:150ms cubic-bezier(0.4,0,0.2,1);
  --anim-smooth:300ms cubic-bezier(0.4,0,0.2,1);
  --anim-slow:500ms cubic-bezier(0.4,0,0.2,1);
  --glass-blur:20px; --glass-bg:rgba(255,255,255,0.08);
  --glass-border:rgba(255,255,255,0.12);
}

MOBILE SHELL:
*, *::before, *::after { box-sizing:border-box; margin:0; padding:0; }
body { margin:0; min-height:100vh; display:flex; justify-content:center;
       align-items:center; background:var(--color-background);
       font-family:var(--font-body,sans-serif); }
.iphone { width:320px; height:650px; background:#000; border-radius:50px; padding:14px;
          box-shadow:0 0 0 4px #2a2a2a, 0 20px 50px rgba(0,0,0,0.7);
          transition:transform var(--anim-smooth); }
.iphone:hover { transform:translateY(-15px); }
.screen { width:100%; height:100%; background:var(--color-background);
          border-radius:40px; overflow:hidden; position:relative; }
.status-bar { display:flex; justify-content:space-between; align-items:center;
              padding:10px 18px; font-size:12px; font-weight:600;
              color:var(--color-text); }
.content { padding:var(--space-md); overflow-y:auto; height:calc(100% - 44px); }

GLASS CARD + BUTTON (always include):
.glass-card { background:var(--glass-bg); backdrop-filter:blur(var(--glass-blur));
              border:1px solid var(--glass-border); border-radius:var(--radius-md);
              padding:var(--space-md); transition:transform var(--anim-smooth),
              box-shadow var(--anim-smooth); }
.glass-card:hover { transform:translateY(-2px); box-shadow:var(--shadow-elevated); }
.btn-primary { background:var(--color-primary); color:#fff; border:none;
               border-radius:var(--radius-sm); padding:var(--space-sm) var(--space-md);
               font-size:var(--font-fluid-sm); cursor:pointer;
               transition:opacity var(--anim-fast), transform var(--anim-fast); }
.btn-primary:hover  { opacity:0.9; transform:translateY(-1px); }
.btn-primary:active { opacity:1;   transform:translateY(0); }

OUTPUT: Return ONLY the ### CSS section wrapped in ```css fences.
"""

JS_SYSTEM = """
You are a Senior Frontend JavaScript Engineer (2026).

Generate clean, modular, production-ready vanilla JavaScript.

REQUIRED MODULES:
1. initClock()        — live time in #currentTime (12h); skip if element absent
2. initNavigation()   — tab/sidebar nav highlighting
3. initAnimations()   — staggered card entrance (IntersectionObserver)
4. initInteractions() — button ripple, card press, swipe hints
5. initTheme()        — apply dynamic CSS vars from app concept palette

Respect prefers-reduced-motion. Use const/let, arrow functions, addEventListener only.
All init inside DOMContentLoaded.

OUTPUT: Return ONLY the ### JS section wrapped in ```js fences.
"""

CRITIC_SYSTEM = """
You are a Senior Code Quality Reviewer and UI/UX Auditor (2026).
Audit the merged HTML+CSS+JS document and return ONLY valid JSON:
{
  "score": 0-100,
  "issues": [
    { "id":"ISSUE_001", "severity":"critical|major|minor",
      "category":"html|css|js|accessibility|performance|ux",
      "description":"...", "location":"...", "fix":"..." }
  ],
  "summary": "..."
}
Return ONLY the JSON — no preamble, no markdown.
"""

FIXER_SYSTEM = """
You are a Senior Full-Stack Engineer and Code Surgeon (2026).
Apply every fix from the audit report to the merged HTML document.
Return a SINGLE, COMPLETE production-ready HTML file starting with <!DOCTYPE html>.
All CSS inside <style> in <head>. All JS inside <script defer> at end of <body>.
Return ONLY the HTML — no explanation, no markdown fences.
"""

BASE_PROMPT = """
Design a minimal, retro command-line interface (CLI) themed website called page.ui.
Retro-terminal aesthetic (green-on-black or amber) + modern minimalism.
Split layout: left prompt/history panel, right preview canvas.
Status bar with model version, last-saved time, keyboard hints.
"""


# ─────────────────────────────────────────────
# Target-Aware Configs
# ─────────────────────────────────────────────

def get_html_system(target: UITarget) -> str:
    return HTML_SYSTEM_WEB if target == UITarget.WEB else HTML_SYSTEM_MOBILE

def get_css_system(target: UITarget) -> str:
    return CSS_SYSTEM_WEB if target == UITarget.WEB else CSS_SYSTEM_MOBILE

def get_ui_requirements(target: UITarget) -> str:
    if target == UITarget.WEB:
        return """STRICT UI REQUIREMENTS:
- Full-browser layout: .app-shell → .topbar + .workspace + .statusbar
- .workspace contains .sidebar and .main-panel
- Design tokens, SVG icons, ARIA roles, Tailwind CDN
- Smooth micro-interactions, real placeholder images
- NO iPhone frame"""
    return """STRICT UI REQUIREMENTS:
- iPhone frame: .iphone → .screen → .status-bar + .content
- Design tokens, SVG icons, ARIA roles, Tailwind CDN
- Smooth micro-interactions, real placeholder images"""


# ─────────────────────────────────────────────
# Merge Helper
# ─────────────────────────────────────────────

def merge_into_html(html: str, css: str, js: str) -> str:
    if '<link rel="stylesheet" href="styles.css">' in html:
        html = html.replace(
            '<link rel="stylesheet" href="styles.css">',
            f'<style>\n{css}\n</style>',
        )
    else:
        html = html.replace('</head>', f'<style>\n{css}\n</style>\n</head>')
    if '<script src="app.js" defer></script>' in html:
        html = html.replace(
            '<script src="app.js" defer></script>',
            f'<script defer>\n{js}\n</script>',
        )
    else:
        html = html.replace('</body>', f'<script defer>\n{js}\n</script>\n</body>')
    return html


# ─────────────────────────────────────────────
# Title Agent
# ─────────────────────────────────────────────

def generate_session_title(prompt: str) -> str:
    raw   = call_title_model(TITLE_SYSTEM, prompt, max_new_tokens=20, temperature=0.1)
    title = raw.strip().strip('"\'').strip()
    for prefix in ("title:", "Title:", "output:", "Output:", "->", "→"):
        if title.lower().startswith(prefix.lower()):
            title = title[len(prefix):].strip()
    words = title.split()
    if len(words) > 4:
        title = " ".join(words[:4])
    elif len(words) < 2:
        prompt_words = [w for w in prompt.split() if len(w) > 3]
        title = " ".join(prompt_words[:2]).title() if prompt_words else "UI Project"
    return title


# ─────────────────────────────────────────────
# UI Target Detection
# ─────────────────────────────────────────────

def detect_ui_target(prompt: str) -> UITarget:
    classification = call_model(
        system_prompt="""You are a UI target classifier.
Return ONLY one word: "mobile" or "web".
Default to "web" if uncertain. "mobile" only for explicit phone/mobile app UIs.""",
        user_prompt=prompt,
        max_new_tokens=5,
        temperature=0.1,
    )
    result = classification.strip().lower()
    if "mobile" in result:
        print("  ✓ LLM-detected: Mobile UI\n")
        return UITarget.MOBILE
    print("  ✓ LLM-detected: Web UI (default)\n")
    return UITarget.WEB


# ─────────────────────────────────────────────
# Cache Reconstruction Helper
# ─────────────────────────────────────────────

def _reconstruct_from_cache(entry: CachedUI) -> str:
    return f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>{entry.concept.get('app_name', 'App')}</title>
  <script src="https://cdn.tailwindcss.com"></script>
  <style>
{entry.css_tokens}
  </style>
</head>
<body>
{entry.html_fragment}
<script defer>
{entry.js_modules}
</script>
</body>
</html>"""


_cache = UICache(cache_dir="./cache")


# ─────────────────────────────────────────────
# Title Pipeline
# ─────────────────────────────────────────────

def run_title_pipeline(prompt: str) -> dict:
    """Runs only the title agent and the UI-target detector."""
    print("\n[Title Pipeline] Starting…")

    print("[0.5/1] ★ Title Agent — generating session title…")
    title = generate_session_title(prompt)
    print(f'    Session title: "{title}"')
    torch.cuda.empty_cache(); gc.collect()

    print("[1/1] ★ UI Target Detector…")
    ui_target = detect_ui_target(prompt)
    torch.cuda.empty_cache(); gc.collect()

    print("✓ Title pipeline complete!")
    return {"title": title, "ui_target": ui_target.value}


# ─────────────────────────────────────────────
# Main UI Pipeline  (cache-integrated, rich prompts)
# ─────────────────────────────────────────────

def run_ui_pipeline(
    custom_prompt: str      = "",
    ui_target:     UITarget = None,
    cache_dir:     str      = "./cache",
) -> tuple[str, dict, dict, UITarget]:
    """
    Runs the full 6-agent pipeline with smart cache integration.
    Returns (final_html, concept, audit, ui_target).

    Cache behaviour per tier:
      "full"      → skip agents 1-4; re-run only Critic + Fixer on cached HTML
      "concept"   → skip agent 1 (Ideation); seed agents 2-4 with cached concept
      "component" → all agents run, prompts include cache guides
      "none"      → cold generation
    """
    global _cache
    if cache_dir != "./cache":
        _cache = UICache(cache_dir=cache_dir)

    prompt = custom_prompt or BASE_PROMPT
    if ui_target is None:
        ui_target = detect_ui_target(prompt)

    target_label    = "Web UI" if ui_target == UITarget.WEB else "Mobile UI"
    html_system     = get_html_system(ui_target)
    css_system      = get_css_system(ui_target)
    ui_requirements = get_ui_requirements(ui_target)

    print(f"\n  Building: {target_label}")
    print("\n[Cache] Looking up prompt…")
    match = _cache.lookup(prompt, ui_target.value)

    # ── FAST PATH ────────────────────────────────────────────────
    if match.found and match.guide_type == "full":
        print("\n[Cache] ⚡ FULL HIT — skipping agents 1-4, using cached HTML")
        concept     = match.entry.concept
        merged      = _reconstruct_from_cache(match.entry)
        concept_str = json.dumps(concept, indent=2)

    else:
        # ── Stage 1: Ideation ─────────────────────────────────────
        if match.found and match.guide_type in ("full", "concept"):
            print("\n[1/6] ★ Ideation Agent — [CACHE: concept reused, skipping LLM]")
            concept     = match.entry.concept
            concept_str = json.dumps(concept, indent=2)
            print(f"    App: {concept.get('app_name', 'Unknown')} — {concept.get('tagline', '')}")
        else:
            print("\n[1/6] ★ Ideation Agent — generating app concept…")
            concept_guide = build_concept_guide(match)
            raw_concept   = call_model(
                system_prompt=IDEATION_SYSTEM,
                user_prompt=prompt + "\nReturn ONLY valid JSON as specified.\n" + concept_guide,
                max_new_tokens=770,
                temperature=0.8,
            )
            concept     = extract_json(raw_concept)
            concept_str = json.dumps(concept, indent=2)
            print(f"    App: {concept.get('app_name', 'Unknown')} — {concept.get('tagline', '')}")
            torch.cuda.empty_cache(); gc.collect()

        # ── Stage 2: HTML ─────────────────────────────────────────
        print("\n[2/6] ★ HTML Agent — generating markup…")
        html_guide  = build_html_guide(match)
        html_prompt = (
            f"App Concept:\n{concept_str}\n\n"
            f"UI Target: {target_label}\n\n"
            f"{ui_requirements}\n\n"
            f"{html_guide}\n\n"
            f"Generate the complete HTML for this {target_label} UI."
        )
        raw_html = call_model(
            system_prompt=html_system,
            user_prompt=html_prompt,
            max_new_tokens=3500,
        )
        html = extract_section(raw_html, "html")
        torch.cuda.empty_cache(); gc.collect()

        # ── Stage 3: CSS ──────────────────────────────────────────
        print("\n[3/6] ★ CSS Agent — generating styles…")
        css_guide  = build_css_guide(match)
        css_prompt = (
            f"App Concept:\n{concept_str}\n\n"
            f"UI Target: {target_label}\n\n"
            f"{ui_requirements}\n\n"
            f"{css_guide}\n\n"
            f"Generate the complete CSS for this {target_label} UI."
        )
        raw_css = call_model(
            system_prompt=css_system,
            user_prompt=css_prompt,
            max_new_tokens=3500,
        )
        css = extract_section(raw_css, "css")
        torch.cuda.empty_cache(); gc.collect()

        # ── Stage 4: JS ───────────────────────────────────────────
        print("\n[4/6] ★ JS Agent — generating interactions…")
        js_guide  = build_js_guide(match)
        js_prompt = (
            f"App Concept:\n{concept_str}\n\n"
            f"UI Target: {target_label}\n\n"
            f"{js_guide}\n\n"
            f"Generate the complete JavaScript for this {target_label} UI.\n"
            f"Implement all 5 required modules: initClock, initNavigation, "
            f"initAnimations, initInteractions, initTheme.\n"
            f"Note: For Web UI, #currentTime may not exist — guard with null check."
        )
        raw_js = call_model(
            system_prompt=JS_SYSTEM,
            user_prompt=js_prompt,
            max_new_tokens=2500,
        )
        js = extract_section(raw_js, "js")

        # ── Merge ─────────────────────────────────────────────────
        print("\n    Merging HTML + CSS + JS…")
        merged = merge_into_html(html, css, js)
        torch.cuda.empty_cache(); gc.collect()

    # ── Stage 5: Critic ───────────────────────────────────────────
    print("\n[5/6] ★ Critic Agent — auditing code quality…")
    raw_audit = call_model(
        system_prompt=CRITIC_SYSTEM,
        user_prompt=(
            f"UI Target: {target_label}\n\n"
            f"Audit the following merged HTML document:\n\n{merged}"
        ),
        max_new_tokens=1500,
        temperature=0.2,
    )
    audit    = extract_json(raw_audit)
    score    = audit.get("score", "N/A")
    issues   = audit.get("issues", [])
    critical = [i for i in issues if i.get("severity") == "critical"]
    print(f"    Score: {score}/100 | Issues: {len(issues)} total, {len(critical)} critical")
    torch.cuda.empty_cache(); gc.collect()

    # ── Stage 6: Fixer ────────────────────────────────────────────
    print("\n[6/6] ★ Fixer Agent — applying patches…")
    fixer_prompt = (
        f"UI TARGET: {target_label}\n\n"
        f"MERGED HTML DOCUMENT:\n{merged}\n\n"
        f"AUDIT REPORT:\n{json.dumps(audit, indent=2)}\n\n"
        f"Apply all fixes and return the single, complete, production-ready HTML file."
    )
    final_html = call_model(
        system_prompt=FIXER_SYSTEM,
        user_prompt=fixer_prompt,
        max_new_tokens=9000,
        temperature=0.3,
    )
    if final_html.startswith("```"):
        final_html = re.sub(r"^```\w*\n?", "", final_html)
        final_html = re.sub(r"```$", "", final_html).strip()

    print("\n✓ UI pipeline complete!")
    torch.cuda.empty_cache(); gc.collect()

    print("\n[Cache] Storing result…")
    _cache.store(
        prompt=prompt,
        ui_target=ui_target.value,
        concept=concept,
        final_html=final_html,
        audit=audit,
    )

    return final_html, concept, audit, ui_target


# ═════════════════════════════════════════════
# PART C — FLASK API  (unchanged from v2)
# ═════════════════════════════════════════════

GENERATION_TIMEOUT_SECONDS = 2 * 60 * 60   # 2 h
JOB_RETENTION_SECONDS      = 3 * 60 * 60   # 3 h
MAX_GENERATION_WORKERS     = 1

app        = Flask(__name__)
jobs: dict[str, dict] = {}
jobs_lock  = threading.Lock()
model_lock = threading.Lock()
executor   = ThreadPoolExecutor(max_workers=MAX_GENERATION_WORKERS)


def now_ts() -> float:
    return time.time()


def safe_job_copy(job: dict) -> dict:
    copied = dict(job)
    if copied.get("status") not in {"failed", "timeout"}:
        copied.pop("traceback", None)
    return copied


def cleanup_old_jobs():
    cutoff = now_ts() - JOB_RETENTION_SECONDS
    with jobs_lock:
        expired = [
            jid
            for jid, job in jobs.items()
            if job.get("status") in {"done", "failed", "timeout", "cancelled"}
            and job.get("finished_at", job.get("created_at", now_ts())) < cutoff
        ]
        for jid in expired:
            jobs.pop(jid, None)


def mark_timed_out_jobs():
    now = now_ts()
    with jobs_lock:
        for job in jobs.values():
            if job.get("status") in {"queued", "running"}:
                if now - job.get("created_at", now) > GENERATION_TIMEOUT_SECONDS:
                    job.update({
                        "status":      "timeout",
                        "finished_at": now,
                        "message":     "Generation exceeded the 2-hour timeout.",
                    })


def update_job(job_id: str, **updates):
    with jobs_lock:
        job = jobs.get(job_id)
        if job:
            job.update(updates)


def get_job(job_id: str) -> Optional[dict]:
    mark_timed_out_jobs()
    with jobs_lock:
        job = jobs.get(job_id)
        return safe_job_copy(job) if job else None


def run_generation_job(job_id: str, prompt: str, ui_target: Optional[UITarget]):
    update_job(job_id, status="running", started_at=now_ts(), message="UI generation started.")
    try:
        with model_lock:
            final_html, concept, audit, ui_target_result = run_ui_pipeline(
                custom_prompt=prompt,
                ui_target=ui_target,
            )
        update_job(
            job_id,
            status      = "done",
            finished_at = now_ts(),
            message     = "UI generation complete.",
            result      = {
                "html":      final_html,
                "concept":   concept,
                "audit":     audit,
                "ui_target": ui_target_result.value,
                "title":     concept.get("app_name", ""),
            },
        )
    except Exception as exc:
        update_job(
            job_id,
            status      = "failed",
            finished_at = now_ts(),
            message     = str(exc),
            traceback   = traceback.format_exc(),
        )
        print(f"[ERROR] Job {job_id} failed: {exc}")
        print(traceback.format_exc())


# ── Routes ────────────────────────────────────────────────────────────────────

@app.route("/health", methods=["GET"])
def health():
    mark_timed_out_jobs()
    with jobs_lock:
        active_jobs = sum(1 for j in jobs.values() if j.get("status") in {"queued", "running"})
        total_jobs  = len(jobs)
    return jsonify({
        "status":          "ok",
        "active_jobs":     active_jobs,
        "total_jobs":      total_jobs,
        "timeout_seconds": GENERATION_TIMEOUT_SECONDS,
    })


@app.route("/cache/stats", methods=["GET"])
def cache_stats():
    return jsonify(_cache.stats())


@app.route("/generate/title", methods=["POST"])
def generate_title():
    """Lightweight endpoint — returns session title + detected UI target.

    Request  { "prompt": "..." }
    Response { "title": "...", "ui_target": "web"|"mobile" }
    """
    data = request.get_json(silent=True) or {}
    if "prompt" not in data:
        return jsonify({"error": "prompt is required"}), 400
    try:
        with model_lock:
            result = run_title_pipeline(str(data["prompt"]))
        return jsonify(result)
    except Exception as e:
        print(f"[ERROR] Title pipeline failed: {e}")
        print(traceback.format_exc())
        return jsonify({"error": str(e)}), 500


@app.route("/generate/ui", methods=["POST"])
def generate_ui_start():
    """Start an async UI generation job.

    Request  { "prompt": "...", "ui_target": "web"|"mobile" }
    Response 202  { "job_id": "...", "status": "queued", "poll_url": "..." }
    """
    cleanup_old_jobs()

    data = request.get_json(silent=True) or {}
    if "prompt" not in data:
        return jsonify({"error": "prompt is required"}), 400

    prompt        = str(data["prompt"])
    ui_target_str = data.get("ui_target")
    ui_target     = (
        UITarget.MOBILE if ui_target_str == "mobile"
        else UITarget.WEB if ui_target_str == "web"
        else None
    )

    job_id = uuid.uuid4().hex
    with jobs_lock:
        jobs[job_id] = {
            "job_id":     job_id,
            "status":     "queued",
            "created_at": now_ts(),
            "message":    "Generation queued.",
            "ui_target":  ui_target.value if ui_target else None,
        }

    executor.submit(run_generation_job, job_id, prompt, ui_target)

    return jsonify({
        "job_id":          job_id,
        "status":          "queued",
        "poll_url":        f"/generate/ui/{job_id}",
        "timeout_seconds": GENERATION_TIMEOUT_SECONDS,
    }), 202


@app.route("/generate/ui/<job_id>", methods=["GET"])
def generate_ui_status(job_id: str):
    """Poll the status of an async job."""
    job = get_job(job_id)
    if not job:
        return jsonify({"error": "job not found or expired"}), 404
    return jsonify(job)


@app.route("/generate/ui/<job_id>/result", methods=["GET"])
def generate_ui_result(job_id: str):
    """Fetch the final result of a completed job."""
    job = get_job(job_id)
    if not job:
        return jsonify({"error": "job not found or expired"}), 404
    if job.get("status") != "done":
        return jsonify({
            "job_id":  job_id,
            "status":  job.get("status"),
            "message": job.get("message", "Generation is not finished yet."),
        }), 202
    return jsonify(job.get("result", {}))


@app.route("/generate/ui-sync", methods=["POST"])
def generate_ui_sync():
    """Blocking (synchronous) fallback — same as async but waits for completion.

    Request  { "prompt": "...", "ui_target": "web"|"mobile" }
    Response { "html": "...", "concept": {...}, "audit": {...}, ... }
    """
    data = request.get_json(silent=True) or {}
    if "prompt" not in data:
        return jsonify({"error": "prompt is required"}), 400

    prompt        = str(data["prompt"])
    ui_target_str = data.get("ui_target")
    ui_target     = (
        UITarget.MOBILE if ui_target_str == "mobile"
        else UITarget.WEB if ui_target_str == "web"
        else None
    )

    try:
        with model_lock:
            final_html, concept, audit, ui_target_result = run_ui_pipeline(
                custom_prompt=prompt,
                ui_target=ui_target,
            )
        return jsonify({
            "html":      final_html,
            "concept":   concept,
            "audit":     audit,
            "ui_target": ui_target_result.value,
            "title":     concept.get("app_name", ""),
        })
    except Exception as exc:
        print(f"[ERROR] Sync UI pipeline failed: {exc}")
        print(traceback.format_exc())
        return jsonify({"error": str(exc)}), 500


# ═════════════════════════════════════════════
# PART D — ngrok + Flask startup  (unchanged)
# ═════════════════════════════════════════════

import requests as _req

ngrok.set_auth_token("39infAR0XBRSwc6lKm9OlZggWe3_4EViH1vb8kModzuEd5mXd")

try:
    ngrok.kill()
    time.sleep(2)
except Exception:
    pass

try:
    tunnels = _req.get("http://127.0.0.1:4040/api/tunnels", timeout=3).json()
    for t in tunnels.get("tunnels", []):
        _req.delete(f"http://127.0.0.1:4040/api/tunnels/{t['name']}", timeout=3)
        print(f"[ngrok] Closed tunnel: {t['name']}")
    time.sleep(1)
except Exception:
    pass

_tunnel    = ngrok.connect(5000, "http")
public_url = getattr(_tunnel, "public_url", str(_tunnel))

print("=" * 50)
print(f"  Kaggle API URL : {public_url}")
print(f"  KAGGLE_API_URL={public_url}")
print("=" * 50)

server_thread = threading.Thread(
    target=lambda: app.run(
        host        = "0.0.0.0",
        port        = 5000,
        debug       = False,
        use_reloader= False,
        threaded    = True,
    ),
    daemon=True,
)
server_thread.start()

print("Kaggle Flask server started ✓")
print("Endpoints:")
print("  GET  /health")
print("  GET  /cache/stats")
print("  POST /generate/title")
print("  POST /generate/ui              → starts async job")
print("  GET  /generate/ui/<job_id>     → polls status")
print("  GET  /generate/ui/<job_id>/result")
print("  POST /generate/ui-sync         → blocking fallback")

try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("Stopping…")
    try:
        ngrok.kill()
    except Exception:
        pass
    executor.shutdown(wait=False, cancel_futures=True)
    print("Stopped.")

Loading processor…


processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

127.0.0.1 - - [11/May/2026 09:10:59] "GET /generate/ui/c76eea0004034cda939942e0e9948f92 HTTP/1.1" 200 -
127.0.0.1 - - [11/May/2026 09:11:14] "GET /generate/ui/c76eea0004034cda939942e0e9948f92 HTTP/1.1" 200 -
127.0.0.1 - - [11/May/2026 09:11:30] "GET /generate/ui/c76eea0004034cda939942e0e9948f92 HTTP/1.1" 200 -
127.0.0.1 - - [11/May/2026 09:11:45] "GET /generate/ui/c76eea0004034cda939942e0e9948f92 HTTP/1.1" 200 -
127.0.0.1 - - [11/May/2026 09:12:00] "GET /generate/ui/c76eea0004034cda939942e0e9948f92 HTTP/1.1" 200 -
127.0.0.1 - - [11/May/2026 09:12:16] "GET /generate/ui/c76eea0004034cda939942e0e9948f92 HTTP/1.1" 200 -
127.0.0.1 - - [11/May/2026 09:12:31] "GET /generate/ui/c76eea0004034cda939942e0e9948f92 HTTP/1.1" 200 -
127.0.0.1 - - [11/May/2026 09:12:47] "GET /generate/ui/c76eea0004034cda939942e0e9948f92 HTTP/1.1" 200 -
127.0.0.1 - - [11/May/2026 09:13:02] "GET /generate/ui/c76eea0004034cda939942e0e9948f92 HTTP/1.1" 200 -
127.0.0.1 - - [11/May/2026 09:13:17] "GET /generate/ui/c76eea000

    Score: 78/100 | Issues: 5 total, 1 critical

[6/6] ★ Fixer Agent — applying patches…


127.0.0.1 - - [11/May/2026 09:13:48] "GET /generate/ui/c76eea0004034cda939942e0e9948f92 HTTP/1.1" 200 -
127.0.0.1 - - [11/May/2026 09:14:04] "GET /generate/ui/c76eea0004034cda939942e0e9948f92 HTTP/1.1" 200 -
127.0.0.1 - - [11/May/2026 09:14:19] "GET /generate/ui/c76eea0004034cda939942e0e9948f92 HTTP/1.1" 200 -
127.0.0.1 - - [11/May/2026 09:14:34] "GET /generate/ui/c76eea0004034cda939942e0e9948f92 HTTP/1.1" 200 -
127.0.0.1 - - [11/May/2026 09:14:50] "GET /generate/ui/c76eea0004034cda939942e0e9948f92 HTTP/1.1" 200 -
127.0.0.1 - - [11/May/2026 09:15:05] "GET /generate/ui/c76eea0004034cda939942e0e9948f92 HTTP/1.1" 200 -
127.0.0.1 - - [11/May/2026 09:15:20] "GET /generate/ui/c76eea0004034cda939942e0e9948f92 HTTP/1.1" 200 -
127.0.0.1 - - [11/May/2026 09:15:36] "GET /generate/ui/c76eea0004034cda939942e0e9948f92 HTTP/1.1" 200 -
127.0.0.1 - - [11/May/2026 09:15:51] "GET /generate/ui/c76eea0004034cda939942e0e9948f92 HTTP/1.1" 200 -
127.0.0.1 - - [11/May/2026 09:16:07] "GET /generate/ui/c76eea000


✓ UI pipeline complete!

[Cache] Storing result…
[Cache] ✓ Stored  (id=dd99ece02499c6d0, score=78)


127.0.0.1 - - [11/May/2026 09:33:03] "GET /generate/ui/c76eea0004034cda939942e0e9948f92 HTTP/1.1" 200 -
